
# Task A — Phishing Detection Benchmark

This notebook prepares phishing datasets, trains a suite of models (classical ML + transformers), and exports the best-performing artifact bundle for downstream use in Task B. All outputs are saved under `/kaggle/working/` so they persist within the Kaggle environment.


In [ ]:

# Cell 1 — Install dependencies (quiet to reduce log noise)
!pip install -q pandas numpy scikit-learn xgboost torch transformers datasets tqdm beautifulsoup4 lxml joblib matplotlib seaborn


In [ ]:

# Cell 2 — Imports, seeding, and helpers
import os
import json
import hashlib
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def log(msg: str):
    print(f"[LOG] {msg}")

ARTIFACT_DIR = Path("/kaggle/working/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("/kaggle/working/data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR = Path("/kaggle/working/reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

# Cell 3 — Dataset configuration (update dataset slug as needed)
KAGGLE_DATASET_SLUG = "your-private-dataset-slug"  # TODO: replace with your dataset slug
KAGGLE_ZEFANG_PATH = f"/kaggle/input/{KAGGLE_DATASET_SLUG}/zefang_liu.csv"
KAGGLE_CYRADAR_PATH = f"/kaggle/input/{KAGGLE_DATASET_SLUG}/cyradar.csv"
HF_DOWNLOAD = False  # set True only if internet is enabled

HF_DATASETS = {
    "zefang_liu": {
        "repo_id": "yiyanghkust/PhishingEmail",
        "filename": "zefang_liu.csv",
    },
    "cyradar": {
        "repo_id": "caiquo/cyradar_phishing",
        "filename": "cyradar.csv",
    },
}

RAW_CACHE_DIR = Path("/kaggle/working/data/raw")
RAW_CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

# Cell 4 — Load helpers (local files first, fallback to HF if allowed)
from datasets import load_dataset


def _download_from_hf(target_path: Path, repo_id: str, filename: str) -> Path:
    log(f"Downloading {filename} from Hugging Face repo {repo_id} ...")
    ds = load_dataset("csv", data_files={"file": f"hf://datasets/{repo_id}/{filename}"})
    df = ds["file"].to_pandas()
    target_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(target_path, index=False)
    return target_path


def get_dataset_path(preferred_path: str, fallback_key: str) -> Path:
    path = Path(preferred_path)
    if path.exists():
        log(f"Found dataset at {path}")
        return path
    if HF_DOWNLOAD:
        info = HF_DATASETS[fallback_key]
        target = RAW_CACHE_DIR / info["filename"]
        if target.exists():
            log(f"Using cached Hugging Face download at {target}")
            return target
        return _download_from_hf(target, info["repo_id"], info["filename"])
    raise FileNotFoundError(
        f"Could not locate {preferred_path}. Attach a Kaggle dataset containing the CSV or enable internet and set HF_DOWNLOAD=True."
    )


def load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, encoding="utf-8", errors="ignore").fillna("")


In [ ]:

# Cell 5 — Load CSVs and align schema
zefang_path = get_dataset_path(KAGGLE_ZEFANG_PATH, "zefang_liu")
cyradar_path = get_dataset_path(KAGGLE_CYRADAR_PATH, "cyradar")

zefang_df = load_csv(zefang_path)
cyradar_df = load_csv(cyradar_path)

log(f"zefang_liu columns: {zefang_df.columns.tolist()}")
log(f"cyradar columns: {cyradar_df.columns.tolist()}")


def map_zefang(df: pd.DataFrame) -> pd.DataFrame:
    columns_lower = {col.lower(): col for col in df.columns}
    text_col = columns_lower.get("email text") or columns_lower.get("text") or columns_lower.get("body")
    label_col = columns_lower.get("email type") or columns_lower.get("label")
    if text_col is None or label_col is None:
        raise ValueError("zefang_liu dataset missing required columns")
    text_series = df[text_col].astype(str)
    label_series = df[label_col].astype(str).str.lower()
    label_mapped = label_series.map({"phishing email": 1, "phishing": 1, "1": 1}).fillna(0).astype(int)
    safe_mask = label_series.isin(["safe email", "legitimate", "0"])
    label_mapped = np.where(safe_mask, 0, label_mapped)
    mapped = pd.DataFrame({
        "subject": "",
        "body": text_series,
        "label": label_mapped,
    })
    mapped["source_dataset"] = "zefang_liu"
    mapped["source_type"] = "email"
    return mapped


def map_cyradar(df: pd.DataFrame) -> pd.DataFrame:
    columns_lower = {col.lower(): col for col in df.columns}
    text_col = columns_lower.get("text") or columns_lower.get("body")
    label_col = columns_lower.get("label")
    if text_col is None or label_col is None:
        raise ValueError("cyradar dataset missing required columns")
    mapped = pd.DataFrame({
        "subject": "",
        "body": df[text_col].astype(str),
        "label": df[label_col].astype(str).str.extract(r"(\d)").fillna("0").astype(int),
    })
    mapped["source_dataset"] = "cyradar"
    mapped["source_type"] = "text"
    return mapped

zefang_mapped = map_zefang(zefang_df)
cyradar_mapped = map_cyradar(cyradar_df)

combined_df = pd.concat([zefang_mapped, cyradar_mapped], ignore_index=True)
combined_df["body_is_html"] = combined_df["body"].str.contains(r"<[^>]+>", regex=True)
combined_df.insert(0, "id", combined_df[["subject", "body"]].astype(str).agg(lambda x: hashlib.sha1("||".join(x).encode("utf-8")).hexdigest(), axis=1))
combined_df = combined_df[["id", "source_dataset", "source_type", "subject", "body", "body_is_html", "label"]]

log(f"Combined dataset shape: {combined_df.shape}")
combined_df.head()


In [ ]:

# Cell 6 — Cleaning and normalization helpers
import re
from html import unescape

URL_TOKEN = "<URL>"
NUM_TOKEN = "<NUM>"


def strip_html(text: str) -> str:
    soup = BeautifulSoup(text, "lxml")
    for tag in soup(["script", "style"]):
        tag.decompose()
    return soup.get_text(separator=" ").strip()


def normalize_text(text: str, lower: bool = True, url_token: str = URL_TOKEN, num_token: str = NUM_TOKEN) -> str:
    text = unescape(text)
    if lower:
        text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", url_token, text)
    text = re.sub(r"\b[0-9]+\b", num_token, text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def preprocess_row(row: pd.Series) -> pd.Series:
    body = row["body"] if row["body"] else ""
    if row["body_is_html"]:
        body = strip_html(body)
    clean = normalize_text(body)
    row["body_clean"] = clean
    return row

combined_df = combined_df.apply(preprocess_row, axis=1)
combined_df["hash"] = combined_df[["subject", "body_clean"]].astype(str).agg(lambda x: hashlib.sha1("||".join(x).encode("utf-8")).hexdigest(), axis=1)
initial_size = len(combined_df)
combined_df = combined_df[combined_df["body_clean"].str.len() > 10]
combined_df = combined_df.drop_duplicates(subset="hash")
log(f"Dropped {initial_size - len(combined_df)} rows due to cleaning and deduplication")
combined_df.head()


In [ ]:

# Cell 7 — Exploratory summary
lengths = combined_df["body_clean"].str.len()
summary = {
    "total": len(combined_df),
    "by_dataset": combined_df.groupby("source_dataset").size().to_dict(),
    "label_counts": combined_df["label"].value_counts().to_dict(),
    "html_true": int(combined_df["body_is_html"].sum()),
    "avg_length": float(lengths.mean()),
    "median_length": float(lengths.median()),
}
log(json.dumps(summary, indent=2))

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.hist(lengths, bins=40, color="#1f77b4", edgecolor="black")
plt.title("Distribution of cleaned body length")
plt.xlabel("Length (characters)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:

# Cell 8 — Train/validation/test splits
train_df, temp_df = train_test_split(
    combined_df,
    test_size=0.2,
    stratify=combined_df["label"],
    random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=SEED,
)

for split_name, split_df in {"train": train_df, "val": val_df, "test": test_df}.items():
    split_df.to_csv(DATA_DIR / f"{split_name}.csv", index=False)
    log(f"Saved {split_name} split with shape {split_df.shape}")


In [ ]:

# Cell 9 — TF-IDF features
from sklearn.feature_extraction.text import TfidfVectorizer
from joblib import dump

vectorizer = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    max_features=100_000,
    sublinear_tf=True,
)

X_train = vectorizer.fit_transform(train_df["body_clean"])
X_val = vectorizer.transform(val_df["body_clean"])
X_test = vectorizer.transform(test_df["body_clean"])

dump(vectorizer, ARTIFACT_DIR / "tfidf_vectorizer.joblib")
log("Saved TF-IDF vectorizer")


In [ ]:

# Cell 10 — Classical ML training
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {}

lr = LogisticRegression(C=2.0, class_weight="balanced", max_iter=1000, n_jobs=4)
lr.fit(X_train, train_df["label"])
models["logistic_regression"] = lr

svm = LinearSVC(class_weight="balanced")
svm.fit(X_train, train_df["label"])
models["linear_svc"] = svm

nb = MultinomialNB()
nb.fit(X_train, train_df["label"])
models["multinomial_nb"] = nb

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)
rf.fit(X_train, train_df["label"])
models["random_forest"] = rf

xgb = XGBClassifier(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=max(1.0, (train_df["label"] == 0).sum() / max(1, (train_df["label"] == 1).sum())),
    tree_method="hist",
    n_jobs=-1,
    random_state=SEED,
)
xgb.fit(X_train, train_df["label"])
models["xgboost"] = xgb

for name, model in models.items():
    dump(model, ARTIFACT_DIR / f"{name}.joblib")
    log(f"Saved {name} model")


In [ ]:

# Cell 11 — Evaluate classical models
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
)
import matplotlib.pyplot as plt
import seaborn as sns

leaderboard_rows = []

for name, model in models.items():
    if hasattr(model, "predict_proba"):
        val_scores = model.predict_proba(X_val)[:, 1]
        test_scores = model.predict_proba(X_test)[:, 1]
    else:
        val_scores = model.decision_function(X_val)
        test_scores = model.decision_function(X_test)
        val_scores = (val_scores - val_scores.min()) / (val_scores.max() - val_scores.min() + 1e-6)
        test_scores = (test_scores - test_scores.min()) / (test_scores.max() - test_scores.min() + 1e-6)

    val_preds = (val_scores >= 0.5).astype(int)
    test_preds = (test_scores >= 0.5).astype(int)

    acc = accuracy_score(val_df["label"], val_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(val_df["label"], val_preds, average="binary", zero_division=0)
    roc_auc = roc_auc_score(val_df["label"], val_scores)
    pr_auc = average_precision_score(val_df["label"], val_scores)

    leaderboard_rows.append({
        "model": name,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    })

    cm = confusion_matrix(test_df["label"], test_preds)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    precision_curve, recall_curve, _ = precision_recall_curve(test_df["label"], test_scores)
    plt.figure(figsize=(4, 3))
    plt.plot(recall_curve, precision_curve, label=name)
    plt.title(f"Precision-Recall Curve — {name}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()
    plt.tight_layout()
    plt.show()

leaderboard_df = pd.DataFrame(leaderboard_rows).sort_values(by=["f1", "pr_auc"], ascending=False)
leaderboard_df

leaderboard_df.to_csv(REPORT_DIR / "leaderboard_classical.csv", index=False)
log("Saved classical leaderboard to reports")


In [ ]:

# Cell 12 — Transformer baseline with Hugging Face Trainer
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

max_length = 256

train_dataset = Dataset.from_pandas(train_df[["body_clean", "label"]].rename(columns={"body_clean": "text"}), preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["body_clean", "label"]].rename(columns={"body_clean": "text"}), preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["body_clean", "label"]].rename(columns={"body_clean": "text"}), preserve_index=False)


def tokenize_function(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=max_length)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

training_args = TrainingArguments(
    output_dir=str(ARTIFACT_DIR / "distilbert"),
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
transformer_metrics = trainer.evaluate(test_dataset)
log(f"Transformer test metrics: {transformer_metrics}")

trainer.save_model(ARTIFACT_DIR / "distilbert")
tokenizer.save_pretrained(ARTIFACT_DIR / "distilbert")
log("Saved transformer model and tokenizer")


In [ ]:

# Cell 13 — Optional LLM zero/few-shot evaluation (disabled by default)
import time
import textwrap

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
SAMPLE_SIZE = 500

if OPENAI_API_KEY:
    try:
        import openai
    except ImportError:
        !pip install -q openai
        import openai

    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    sample = val_df.sample(min(SAMPLE_SIZE, len(val_df)), random_state=SEED)
    predictions = []
    for _, row in tqdm(sample.iterrows(), total=len(sample), desc="LLM inference"):
        prompt = textwrap.dedent(f"""
        Decide if the following message is phishing or legitimate. Respond with '1' for phishing, '0' for legitimate.

        Message:
        {row['body_clean']}
        """)
        response = client.responses.create(
            model="gpt-4o-mini",
            input=prompt,
        )
        content = response.output_text.strip()
        label = 1 if content.startswith("1") else 0
        predictions.append(label)
        time.sleep(0.5)
    llm_preds = np.array(predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(sample["label"], llm_preds, average="binary", zero_division=0)
    acc = accuracy_score(sample["label"], llm_preds)
    log(f"LLM sample metrics — accuracy: {acc:.3f}, precision: {precision:.3f}, recall: {recall:.3f}, f1: {f1:.3f}")
    sample.assign(llm_pred=llm_preds).to_csv(REPORT_DIR / "llm_predictions.csv", index=False)
else:
    log("OPENAI_API_KEY not provided. Skipping LLM zero/few-shot evaluation.")


In [ ]:

# Cell 14 — Select best model and export bundle
import shutil

best_row = leaderboard_df.sort_values(by=["f1", "pr_auc"], ascending=False).iloc[0]
best_name = best_row["model"]
log(f"Best classical model based on validation F1: {best_name}")

bundle_dir = ARTIFACT_DIR / "best_model"
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

preproc_config = {
    "seed": SEED,
    "vectorizer": "vectorizer.joblib",
    "model": "model.joblib",
    "model_type": best_name,
    "transformer": False,
}

transformer_path = ARTIFACT_DIR / "distilbert"
if transformer_path.exists():
    preproc_config["transformer_path"] = "../distilbert"

with open(bundle_dir / "preproc_config.json", "w") as f:
    json.dump(preproc_config, f, indent=2)

shutil.copy(ARTIFACT_DIR / "tfidf_vectorizer.joblib", bundle_dir / "vectorizer.joblib")
shutil.copy(ARTIFACT_DIR / f"{best_name}.joblib", bundle_dir / "model.joblib")

log(f"Exported best model bundle to {bundle_dir}")
